# QLoRA on Ascend NPU

This notebook fine-tunes a chat model with **QLoRA — 4-bit NF4 base + LoRA adapters** — on a Huawei Ascend NPU (e.g. `Ascend910B4`) using `transformers` + `peft` + `torch_npu` + the community `bitsandbytes-npu-beta` fork. It is the NPU companion of [`qlora-comprehensive-tutorial.ipynb`](./qlora-comprehensive-tutorial.ipynb), which uses `training_hub` on CUDA.

> **Why a separate NPU notebook?** `training_hub` targets NVIDIA GPUs via `torch.cuda`, `bitsandbytes` (CUDA build), and `unsloth`. None of those run on Ascend today. Mainline `bitsandbytes` also removed its Ascend backend from the v0.46+ install docs — upstream re-add is tracked in [`bitsandbytes-foundation/bitsandbytes#1695`](https://github.com/bitsandbytes-foundation/bitsandbytes/pull/1695). Until that merges, the practical Ascend path is the community fork [`SlightwindSec/bitsandbytes`](https://github.com/SlightwindSec/bitsandbytes), published as `bitsandbytes-npu-beta` on PyPI.

**Support caveat.** `bitsandbytes-npu-beta==0.45.3` (2025-03-18) was built against roughly `torch_npu 2.1 – 2.4` / `CANN 8.0.RC3 – 8.1`. The Alauda `PyTorch CANN` workbench image ships `torch_npu 2.9.0` / `CANN 8.5.0`. `torch_npu` keeps loose ABI compatibility across `2.x`, but this specific combination is **not officially validated** by the fork's author. If import fails with `undefined symbol` or a dtype mismatch, pin the workbench to an older `PyTorch CANN` image (CANN 8.1 / torch_npu 2.4 range) instead.

**Runtime prerequisites**

- Ascend driver + CANN runtime installed on the node.
- Workbench image: `PyTorch CANN` (Python 3.12, `torch_npu`, `transformers`, `peft`) on an **aarch64** node — `bitsandbytes-npu-beta` ships only a `manylinux2014_aarch64` wheel.
- At least 1 NPU exposed as `huawei.com/Ascend910B4` (or your cluster's resource name) with ≥ 8 GB HBM available.
- Persistent storage for base model + adapter checkpoints.

## When to use QLoRA on NPU

- Base model too large to hold in bf16 on the NPUs you have (e.g. Qwen2.5-14B on a single 32 GB Ascend910B).
- You want a small, cheap adapter you can ship alongside the frozen base model.
- You're okay trading a little quality for a large memory saving — QLoRA is memory-efficient LoRA, not a full-parameter tune.

If the base model already fits in bf16, use plain LoRA (`load_in_4bit=False`) — you skip the fork dependency entirely and stay on upstream `transformers + peft`.

## Step 1 — Install `bitsandbytes-npu-beta`

One-off install into the workbench venv. Everything else (`transformers`, `peft`, `accelerate`, `trl`, `datasets`) comes from the bundled runtime.

```bash
pip install --no-cache-dir bitsandbytes-npu-beta==0.45.3
```

The version pin is deliberate — `0.45.3` is the last (as of Jan 2026) published tag of the community fork. Don't upgrade past it without checking [the PyPI release page](https://pypi.org/project/bitsandbytes-npu-beta/); a newer release may or may not exist.

## Step 2 — Environment check

**Import order matters.** `torch_npu` must be imported **before** `bitsandbytes` so the fork detects the NPU backend.

In [ ]:
import os
# Cuts fragmentation on LoRA / QLoRA workloads.
os.environ.setdefault("PYTORCH_NPU_ALLOC_CONF", "expandable_segments:True")

import torch
import torch_npu  # noqa: F401 — must precede bitsandbytes import
import bitsandbytes as bnb
import transformers, peft

print("torch          :", torch.__version__)
print("torch_npu      :", torch_npu.__version__)
print("bitsandbytes   :", bnb.__version__)
print("transformers   :", transformers.__version__)
print("peft           :", peft.__version__)
assert torch.npu.is_available(), "no NPU visible — check the device plugin and resource limits"
torch.npu.set_device(0)
print("device 0       :", torch.npu.get_device_name(0))

## Step 3 — Parameters

Defaults use a small base and a short training so the notebook completes in a few minutes on a single NPU. Bump `MODEL_PATH`, `NUM_EPOCHS`, and `MAX_SEQ_LEN` for real runs.

In [ ]:
MODEL_PATH = os.environ.get("HF_MODEL_DIR", "/opt/app-root/src/models/Qwen2.5-0.5B-Instruct")
DATA_PATH = os.environ.get("DATA_PATH", "/opt/app-root/src/datasets/chat/train.jsonl")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/opt/app-root/src/qlora-npu-output")

# LoRA
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
NUM_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACC_STEPS = 4              # effective batch = 8
LEARNING_RATE = 2e-4            # QLoRA can use LR much higher than full fine-tune
MAX_SEQ_LEN = 512
WARMUP_STEPS = 20
LOGGING_STEPS = 10
SAVE_STEPS = 200
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 4 — Data format

Chat JSONL — one conversation per line under `messages`:

```json
{"messages": [
  {"role": "system",    "content": "You are a helpful assistant."},
  {"role": "user",      "content": "What is machine learning?"},
  {"role": "assistant", "content": "Machine learning is ..."}
]}
```

The next cell falls back to a tiny built-in synthetic dataset if `DATA_PATH` doesn't exist — the notebook stays runnable as a smoke test.

In [ ]:
import json
from pathlib import Path

if not Path(DATA_PATH).exists():
    fallback = Path(OUTPUT_DIR) / "synthetic_chat.jsonl"
    with fallback.open("w") as f:
        for _ in range(32):
            json.dump({"messages": [
                {"role": "system",    "content": "You are a helpful assistant."},
                {"role": "user",      "content": "Hello, how are you?"},
                {"role": "assistant", "content": "I am well — how can I help?"},
            ]}, f); f.write("\n")
    DATA_PATH = str(fallback)
    print(f"data not found; wrote synthetic fallback to {DATA_PATH}")
print("data:", DATA_PATH)

## Step 5 — Load the base model in 4-bit

`BitsAndBytesConfig` works unchanged against `bitsandbytes-npu-beta` — no `bnb.enable_ascend()` bootstrap needed. Route the model to `npu:0` via `device_map`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": "npu:0"},
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",  # sdpa works on most CANN 8.x builds; eager is the portable default
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6 — Tokenize the chat corpus

Apply the tokenizer's chat template and tokenize each conversation to `input_ids` + `labels`. For QLoRA-style SFT, the whole conversation contributes to loss (Alauda's `training_hub` default is assistant-only masking — the recipe here uses full-turn loss for portability; adapt if you need instruction-only masking).

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=DATA_PATH, split="train")

def render(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    ids = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN,
                    padding="max_length", add_special_tokens=False)
    ids["labels"] = ids["input_ids"].copy()
    return ids

packed = raw.map(render, remove_columns=raw.column_names)
print("samples:", len(packed), "of length", MAX_SEQ_LEN)

## Step 7 — Train

Use plain `Trainer` with `bf16=True` and `optim="adamw_torch"`. Do **not** use `paged_adamw_8bit` — that path relies on CUDA kernels not shipped in the NPU fork.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=True, fp16=False,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to=[],
    remove_unused_columns=False,
    dataloader_pin_memory=False,  # NPU has no CUDA pinned host memory
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=packed,
    data_collator=collator,
)

train_result = trainer.train()
print(train_result.metrics)

## Step 8 — Save the LoRA adapter

Save only the adapter weights (a few MB) — the 4-bit base is unchanged and lives in the base-model directory. To serve the tuned model, re-attach the adapter to the fp16/bf16 base at inference time.

In [ ]:
adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved:", adapter_dir)
print("contents:", sorted(os.listdir(adapter_dir)))

## Step 9 (optional) — Reload adapter for inference

Load the frozen bf16 base and attach the trained adapter for a quick generation smoke. For real inference workflows you'd typically merge the adapter (`PeftModel.merge_and_unload()`) once, save the merged model, and serve the merged HF checkpoint.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
base = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16).to("npu")
tuned = PeftModel.from_pretrained(base, adapter_dir).to("npu")
tuned.eval()

prompt = tok.apply_chat_template(
    [{"role": "user", "content": "Hello, how are you?"}],
    tokenize=False, add_generation_prompt=True,
)
inputs = tok(prompt, return_tensors="pt").to("npu")
with torch.no_grad():
    out = tuned.generate(**inputs, max_new_tokens=32, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

## Notes and gotchas on NPU QLoRA

- **Import order.** `import torch_npu` **before** `import bitsandbytes`. If bnb is imported first, it won't register the NPU backend and 4-bit loading will fall over.
- **`bitsandbytes-npu-beta` version pinning.** `0.45.3` (2025-03-18) is the last known-good published tag. `pip install bitsandbytes` (unpinned) resolves to the upstream CUDA build — always specify the fork's PyPI name.
- **CANN / torch_npu compatibility.** The fork's wheel was built against `torch_npu 2.1 – 2.4`. If import fails on newer stacks (e.g. `torch_npu 2.9` / CANN 8.5), pin the workbench to a CANN 8.1-era `PyTorch CANN` image.
- **Optimizer.** Use `adamw_torch`. `paged_adamw_8bit` and other bnb-specific optimizers rely on CUDA kernels not present in the NPU fork.
- **CPU offload.** Do **not** set `llm_int8_enable_fp32_cpu_offload=True` — the fork does not document offload as working.
- **Attention backend.** `attn_implementation="eager"` is the portable default. `sdpa` usually works on CANN 8.x. `flash_attention_2` is CUDA-only.
- **Precision.** Use `bf16`. Avoid `fp16` on the compute path — Ascend910B prefers bf16 and mixing it with the fork's dequant kernels has been reported to trip recompilation.
- **aarch64 only.** The fork ships only a `manylinux2014_aarch64` wheel; there is no x86 wheel.
- **Data loader.** Set `dataloader_pin_memory=False`; NPU has no CUDA pinned host memory.
- **Path to native support.** Track [`bitsandbytes-foundation/bitsandbytes#1695`](https://github.com/bitsandbytes-foundation/bitsandbytes/pull/1695) — once merged upstream, the fork can be retired and mainline `bitsandbytes` with `--index-url` for a matching wheel becomes the recommended path.